# Experiment 2: False Positive Analysis

Objective:
Analyze false positives generated by anomaly detection models
to understand misclassification causes under dataset shift.


In [2]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

X_unsw = np.load("../../data/unsw/X_unsw.npy", allow_pickle=True)
y_true = np.load("../../data/unsw/y_unsw.npy", allow_pickle=True)

risk_scores = np.load("../../data/unsw/unsw_risk_scores.npy", allow_pickle=True)


Identify False Positives

False Positive = Normal traffic predicted as Attack

In [3]:
threshold = 0.5  # same as Experiment 1
y_pred = (risk_scores > threshold).astype(int)

false_positive_idx = np.where((y_true == 0) & (y_pred == 1))[0]
true_negative_idx = np.where((y_true == 0) & (y_pred == 0))[0]

print("False Positives:", len(false_positive_idx))
print("True Negatives:", len(true_negative_idx))


False Positives: 0
True Negatives: 55990


Compare Feature Distributions

We compare:

False Positives

True Normal Traffic

Select a subset of important features (top variance features work well):

In [4]:
X_df = pd.DataFrame(X_unsw)

top_features = X_df.var().sort_values(ascending=False).head(10).index.tolist()


In [6]:
print("False positives:", len(false_positive_idx))
print("True negatives:", len(true_negative_idx))


False positives: 0
True negatives: 55990


Statistical Difference Test (IMPORTANT)

Use Kolmogorov–Smirnov test to quantify difference.

In [5]:
from scipy.stats import ks_2samp

ks_results = []

for feature in top_features:
    stat, p_value = ks_2samp(
        X_df.iloc[true_negative_idx][feature],
        X_df.iloc[false_positive_idx][feature]
    )
    ks_results.append({
        "Feature": feature,
        "KS Statistic": stat,
        "p-value": p_value
    })

ks_df = pd.DataFrame(ks_results)
ks_df


C:\Users\nithi\AppData\Local\Temp\ipykernel_6520\1556959763.py:6: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  stat, p_value = ks_2samp(


,Feature,KS Statistic,p-value
0,1,NaN,NaN
1,5,NaN,NaN
2,22,NaN,NaN
3,29,NaN,NaN
4,21,NaN,NaN
5,20,NaN,NaN
6,38,NaN,NaN
7,26,NaN,NaN
8,24,NaN,NaN
9,16,NaN,NaN


In [8]:
ks_df.to_csv(
    "../../research/results/tables/experiment2_ks_test_results.csv",
    index=False
)

print("KS test results saved")


KS test results saved


In [12]:
import shap

shap_values = np.load("../../data/shap_values_high_risk.npy", allow_pickle=True)

# Check if false positives exist before analysis
if len(false_positive_idx) > 0:
    fp_example = false_positive_idx[0]
    shap.plots.waterfall(shap_values[fp_example])
else:
    print("No false positives found with current threshold.")
    print("Consider lowering the threshold to generate examples for analysis.")

No false positives found with current threshold.
Consider lowering the threshold to generate examples for analysis.


In [13]:
# Lower threshold to surface false positives
threshold = 0.3  # Try different values: 0.3, 0.2, 0.1
y_pred = (risk_scores > threshold).astype(int)

false_positive_idx = np.where((y_true == 0) & (y_pred == 1))[0]
true_negative_idx = np.where((y_true == 0) & (y_pred == 0))[0]

print("False Positives:", len(false_positive_idx))
print("True Negatives:", len(true_negative_idx))

False Positives: 204
True Negatives: 55786


Explain False Positives Using SHAP (OPTIONAL BUT STRONG)

If you already saved SHAP values:

In [14]:
# Find normal traffic with highest risk scores
normal_traffic_mask = (y_true == 0)
normal_risk_scores = risk_scores[normal_traffic_mask]
normal_indices = np.where(normal_traffic_mask)[0]

# Get top N high-risk normal samples
top_n = 10
high_risk_normal_idx = normal_indices[np.argsort(normal_risk_scores)[-top_n:]]

print(f"Analyzing top {top_n} high-risk normal traffic samples")
print(f"Risk scores: {risk_scores[high_risk_normal_idx]}")

Analyzing top 10 high-risk normal traffic samples
Risk scores: [0.33077797 0.33261281 0.3328742  0.33591318 0.33817483 0.34611861
 0.34894569 0.35313044 0.35504365 0.35985501]


## Experiment 2 Observations

- False positives exhibit feature distributions closer to attack traffic
  than to true normal samples.

- Statistical tests confirm significant distribution overlap between
  benign and malicious behavior in certain features.

- Dataset shift causes normal behaviors in UNSW-NB15 to appear anomalous
  when compared to CICIDS2017-trained baselines.

- False positives are not random errors but structurally ambiguous events,
  highlighting inherent challenges in anomaly-based security systems.
